# CNIG

In [1]:
import requests
from bs4 import BeautifulSoup
import time

In [ ]:
with requests.Session() as session:
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/120 Safari/537.36"
        ),
        "Referer": "https://centrodedescargas.cnig.es/",
    })

    payload = {
        "g-recaptcha-response": "",
        "sesCodLic": "",
        "codLicAnt": "LIGWxxxxxxxx",
        "errDescarga": "",
        "arcSecDesc": "",
        "pagActual": "3",
        "totalFilesDesc": "1011",
        "totalMbDescManual": "91568,8",
    }

    response = session.post(
        url='https://centrodedescargas.cnig.es/CentroDescargas/listaDescManual',
        data=payload,
        cookies={'JSESSIONID': 'xxxx'},
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        timeout=30
    )

In [14]:
soup = BeautifulSoup(response.text, "html.parser")

rows = soup.find_all("tr")

row_ids = [
    int(row.get("id").strip('tr_'))
    for row in rows
    if row.get("id") is not None
]

print(row_ids)

[12641994, 12641995, 12641996, 12641997, 12641998, 12641999, 12642000, 12642001, 12642002, 12642003, 12642004, 12642005, 12642006, 12642007, 12642008, 12642009, 12642010, 12642011, 12642012, 12642013]


In [ ]:
with requests.Session() as session:
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/120 Safari/537.36"
        ),
        "Referer": "https://centrodedescargas.cnig.es/",
    })

    status = 200
    page = 1

    while status == 200:
        print(f"page: {page} of {1011/20}")

        payload = {
            "g-recaptcha-response": "",
            "sesCodLic": "",
            "codLicAnt": "LIGWxxxxxxxxx",
            "errDescarga": "",
            "arcSecDesc": "",
            "pagActual": page,
            "totalFilesDesc": "1011",
            "totalMbDescManual": "91568,8",
        }

        response = session.post(
            url='https://centrodedescargas.cnig.es/CentroDescargas/listaDescManual',
            data=payload,
            cookies={'JSESSIONID': 'xxxxxxx'},
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            timeout=30
        )

        status = response.status_code
        print(f"status: {status}")
        
        if status != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        rows = soup.find_all("tr")

        row_ids = [
            f"https://centrodedescargas.cnig.es/CentroDescargas/detalleArchivo?sec={row.get("id").strip('tr_')}"
            for row in rows
            if row.get("id") is not None
        ]

        with open('cnig-id-urls.txt', 'a') as f:
            f.write("\n".join(row_ids))
            f.write("\n")

        page += 1
        time.sleep(1)
        

I realize after trying this method that CNIG offers a file for OpenWebStart to download hundreds or thousands of files automatically. It was pretty slow but was easier than this method.

## Image split

In order to fine-tune a YOLO model I need to divide the original .tif files in multiple 1024 x 1024 images to label them on the X-Anylabeling software.

Only need to select a couple of this .tif files because each of them convert in almost 400 of individual 1024 x 1024 images.

In [ ]:
# Divide selected .tif images in 1024 x 1024 tiles

import os
import cv2
import numpy as np
import rasterio
from rasterio.windows import Window

input_folder = r"X:\datasets\outdoor-car-detection-satellite\original"
output_folder = r"X:\datasets\outdoor-car-detection-satellite\anotated-images"
os.makedirs(output_folder, exist_ok=True)

tile_size = 1024
stride = 1024 # No overlapping

for tif_file in os.listdir(input_folder):

    tif_path = os.path.join(input_folder, tif_file)
    base_name = os.path.splitext(tif_file)[0]

    with rasterio.open(tif_path) as src:
        width, height = src.width, src.height

        tile_id = 0
        for y in range(0, height - tile_size, stride):
            for x in range(0, width - tile_size, stride):
                window = Window(x, y, tile_size, tile_size)

                img = src.read([1, 2, 3], window=window)

                # 8-bit norm
                if img.dtype != np.uint8:
                    img = (
                        (img - img.min()) / (img.max() - img.min() + 1e-5) * 255
                    ).astype(np.uint8)

                img = np.moveaxis(img, 0, -1)
                img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

                if np.mean(img_bgr) < 10:
                    continue

                out_name = f"{base_name}_tile_{tile_id}.png"
                cv2.imwrite(os.path.join(output_folder, out_name), img_bgr)
                tile_id += 1


In [ ]:
# Delete images that are not anotated

import os
import json

base_dir = r'X:\datasets\outdoor-car-detection-satellite\anotated-images'

png_files = os.listdir(base_dir)
png_files = [file for file in png_files if file.endswith('.png')]

i = 0
for p_file in png_files:
    json_conv = p_file.split('.')[0] + ".json"

    if not os.path.exists(os.path.join(base_dir, json_conv)):
        os.remove(os.path.join(base_dir, p_file))


In [ ]:
# JSON file -> YOLO TXT file

base_dir = r'X:\datasets\outdoor-car-detection-satellite\anotated-images'

IMG_WIDTH = 1024
IMG_HEIGHT = 1024

for i, file_name in enumerate(os.listdir(base_dir)):
    if not file_name.endswith(".json"):
        continue

    json_path = os.path.join(base_dir, file_name)
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    txt_path = os.path.join(base_dir, txt_name)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    yolo_lines = []

    for shape in data.get("shapes", []):
        # Only OBB/rotation objects
        if shape.get("shape_type") in ["rotation", "polygon"]:
            label = shape.get("label", "").lower()

            # Only want 1 unified class: vehicle
            if "vehicle" in label:
                class_id = 0
                points = shape.get("points", [])

                if len(points) == 4:
                    normalized_points = []
                    for px, py in points:
                        # For YOLO format we need to normalize points
                        nx = max(0.0, min(1.0, px / IMG_WIDTH))
                        ny = max(0.0, min(1.0, py / IMG_HEIGHT))
                        normalized_points.extend([f"{nx:.6f}", f"{ny:.6f}"])

                    # YOLO OBB format: class_id x1 y1 x2 y2 x3 y3 x4 y4
                    line = f"{class_id} " + " ".join(normalized_points)
                    yolo_lines.append(line)

                else:
                    print("[ERROR] - 4 points not found")

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_lines))


In [ ]:
# 80% train - 20% val
import random
import shutil

dataset_dir = r'X:\datasets\outdoor-car-detection-satellite'

png_files = os.listdir(base_dir)
png_files = [file for file in png_files if file.endswith('.png')]

random.shuffle(png_files)

for i, png_path in enumerate(png_files):
    print(f"{i}/{len(png_files)}")
    out_dir = "train"
    if i > int(len(png_files))*0.8:
        out_dir = "val"

    txt_path = os.path.splitext(png_path)[0] + ".txt"

    shutil.copy(os.path.join(dataset_dir, 'anotated-images', png_path), os.path.join(dataset_dir, out_dir, png_path))
    shutil.copy(os.path.join(dataset_dir, 'anotated-images', txt_path), os.path.join(dataset_dir, out_dir, txt_path))

## VRT

In order to load the images into a single file and later use them to display them on a map on a website, I need to convert the 1,011 individual `GeoTIFF` files into a single `pmtiles` file. To do this, I must follow these steps:

1. Reproject all my satellite images to ESP:3857; this projection is widely used on the web (Google Maps, OpenStreetMaps, ArcGIS, etc).

```bash
    python convert_to_3857.py
```

2. Create a virtual raster (VRT) by merging all the images.

```bash
    gdalbuildvrt mosaico_3857.vrt ./epsg_3857/*.tif
```

3. Generate the tiles (quadrants) for different zoom levels.

```bash
    set GDAL_PAM_ENABLED=NO
    set GDAL_NUM_THREADS=12
    set GDAL_CACHEMAX=16384

    gdal_translate -of MBTILES -co TILE_FORMAT=WEBP -co QUALITY=80 mosaico_3857.vrt cadiz.mbtiles
    gdaladdo -r average cadiz.mbtiles 2 4 8 16 32 
```

4. Combine all the tiles into a single `pmtiles` file.

```bash
    pmtiles convert cadiz.mbtiles cadiz.pmtiles
```

## Hugging face upload

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj=r"X:\datasets\outdoor-car-detection-satellite\cadiz.mbtiles",
    path_in_repo=r'cadiz.mbtiles',
    repo_id="naim-prog/cadiz-satellite-orthophoto",
    repo_type="dataset",
)

# YOLO

In [11]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 3060


In [2]:
from ultralytics import YOLO
import cv2

In [ ]:
model_11 = YOLO('../model/yolo11x-obb.pt')
model_26 = YOLO('../model/yolo26x-obb.pt')
model_26_custom = YOLO('../model/yolo26x-obb-finetuned.pt')

In [16]:
results = []
for i in range(1, 6):
    results.append(model_26_custom.predict(fr'..\temp\test-images\test-{i}.png', device='cuda:0'))


image 1/1 c:\programming\python\outdoor-car-detection-satellite\src\..\temp\test-images\test-1.png: 1024x1024 191 vehicles, 253.1ms
Speed: 35.1ms preprocess, 253.1ms inference, 6.7ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 c:\programming\python\outdoor-car-detection-satellite\src\..\temp\test-images\test-2.png: 1024x1024 89 vehicles, 89.2ms
Speed: 3.6ms preprocess, 89.2ms inference, 4.1ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 c:\programming\python\outdoor-car-detection-satellite\src\..\temp\test-images\test-3.png: 960x1024 98 vehicles, 83.2ms
Speed: 3.5ms preprocess, 83.2ms inference, 3.1ms postprocess per image at shape (1, 3, 960, 1024)

image 1/1 c:\programming\python\outdoor-car-detection-satellite\src\..\temp\test-images\test-4.png: 960x1024 70 vehicles, 83.4ms
Speed: 3.1ms preprocess, 83.4ms inference, 4.4ms postprocess per image at shape (1, 3, 960, 1024)

image 1/1 c:\programming\python\outdoor-car-detection-satellite\src\..\temp\tes

In [ ]:
for i in range(1, 6):
    anotated_image = results[i-1][0].plot(labels=False, probs=False, conf=False, line_width=2)

    cv2.imwrite(fr'..\temp\test-images\yolo26x-custom_test-{i}.png', anotated_image)
    cv2.imshow('Detecciones', anotated_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

## SAHI

In [1]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="../model/yolo26x-obb-finetuned.pt",
    confidence_threshold=0.3,
    device="cuda:0",  # 'cpu' or 'cuda:0'
)

In [ ]:
# To export a large .tif file with anotations
result = get_sliced_prediction(
    "../images/original/image.tif",
    detection_model,
    slice_height=1024,
    slice_width=1024,
    overlap_height_ratio=150/1024, # 10 meters overlapping
    overlap_width_ratio=150/1024,
    batch_size=16,
    progress_bar=True,
    verbose=2,
)

result.export_visuals(
    export_dir="../images/output",
    rect_th=2,
    text_size=0.4,
    hide_conf=False,
    hide_labels=True,
    file_name='image_output'
)

Performing prediction on 570 slices.


Processing slices: 100%|██████████| 36/36 [01:07<00:00,  1.86s/it]


Slicing performed in 3.976079700005357 seconds.
Prediction performed in 75.32764273729117 seconds.
Postprocessing performed in 0.6804921627044678 seconds.


In [ ]:
import os

BASE_DIR_ORIGINAL = r"X:\datasets\outdoor-car-detection-satellite\original"

original_images_list = [dire for dire in os.listdir(BASE_DIR_ORIGINAL) if dire.endswith('.tif')]
images_list_batched = [
    original_images_list[0:100],
    original_images_list[100:200],
    original_images_list[200:300],
    original_images_list[300:400],
    original_images_list[400:500],
    original_images_list[500:600],
    original_images_list[600:700],
    original_images_list[700:800],
    original_images_list[800:900],
    original_images_list[900:1000],
    original_images_list[1000:],
]

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon
import rasterio
from pyproj import Transformer

BASE_DIR_GEOJSON = r"X:\datasets\outdoor-car-detection-satellite\geojson-models"
TARGET_CRS = "EPSG:4326"

for i_batch, batch_images_list in enumerate(images_list_batched):
    print(f"############### [BATCH] - {i_batch} / {len(images_list_batched)} ###############")
    # If that file already exists, skip to the next one (to running multiple times without overwriting)
    output_geojson = os.path.join(BASE_DIR_GEOJSON, f"yolo26x-obb-finetuned_{i_batch}-{len(images_list_batched)}.geojson")
    if os.path.exists(output_geojson):
        continue

    polygons = []
    confidences = []
    categories = []

    for i, tif_file_name in enumerate(batch_images_list):
        print(f"[IT] - {i} / {len(batch_images_list)}")

        result = get_sliced_prediction(
            os.path.join(BASE_DIR_ORIGINAL, tif_file_name),
            detection_model,
            slice_height=1024,
            slice_width=1024,
            overlap_height_ratio=150/1024, # 10 meters overlapping
            overlap_width_ratio=150/1024,
            batch_size=8,
            progress_bar=True,
            verbose=2,
        )

        with rasterio.open(os.path.join(BASE_DIR_ORIGINAL, tif_file_name)) as src:
            transform = src.transform
            src_crs = src.crs

            transformer = Transformer.from_crs(src_crs, TARGET_CRS, always_xy=True)

            for object_prediction in result.object_prediction_list:
                try:
                    # With object.bbox it returns maxx, minx, maxy, miny. With this we have the 4 corners of the OBB
                    bbox = object_prediction.get_shifted_object_prediction().mask.segmentation[0]

                    pixel_coords = [(bbox[0], bbox[1]), (bbox[2], bbox[3]), (bbox[4], bbox[5]), (bbox[6], bbox[7]), (bbox[0], bbox[1])]
                except Exception as e:
                    print(e.with_traceback())
                    continue

                # tranform the pixels
                native_coords = [rasterio.transform.xy(transform, pt[1], pt[0]) for pt in pixel_coords]
                target_coords = [transformer.transform(pt[0], pt[1]) for pt in native_coords]
                poly = Polygon(target_coords)

                if object_prediction.category.name == 'small vehicle' or \
                    object_prediction.category.name == 'large vehicle' or \
                    object_prediction.category.name == 'vehicle':
                    
                    polygons.append(poly)
                    confidences.append(object_prediction.score.value)
                    categories.append("vehicle")

    gdf = gpd.GeoDataFrame(
        {
            'confidence': confidences,
            'model': 'yolo26x-obb-finetuned',
            'category': categories
        },
        geometry=polygons,
        crs=TARGET_CRS
    )

    gdf.to_file(output_geojson, driver="GeoJSON")
    print(f"Exported to {output_geojson}!")

After transforming all the results for every .tif file we need to join all the GeoJSON files into a single one. For this I used QGIS tool: `merge vector layers` selecting the multiples GeoJSON files and it outputs a single one.

After merging all the GeoJSON files we must transform this one to PMTiles files for better efficiency on web serving.

We make use of tippecanoe tool with the following command:

```bash
tippecanoe -o output.pmtiles
    --layer=vehicle
    --drop-densest-as-needed
    --maximum-zoom=20
    --minimum-zoom=15
    -s EPSG:3857
    input.geojson
```

## Results

In [ ]:
from ultralytics import YOLO

In [ ]:
model_26 = YOLO('../model/yolo26x-obb.pt')
model_26_custom = YOLO('../model/yolo26x-obb-finetuned.pt')

In [58]:
model_26.names[0] = 'vehicle'

In [ ]:
metrics_model_26 = model_26.val(
    data=r"X:\datasets\outdoor-car-detection-satellite\data.yaml",
    split="test",
    classes=[0, 9, 10],
    single_cls=True,
    imgsz=1024,
    batch=16,
    device='cuda:0',
    plots=True
)

Ultralytics 8.4.154  Python-3.12.11 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
YOLO26x-obb summary (fused): 197 layers, 57,559,452 parameters, 0 gradients, 202.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2129.5490.6 MB/s, size: 1627.2 KB)
val: Scanning E:\datasets\outdoor-car-detection-satellite\test.cache... 114 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5s/it 12.3s1.8s
                   all        114       1696       0.87      0.907      0.828       0.74
                 plane         94       1696       0.87      0.907      0.828       0.74
Speed: 5.8ms preprocess, 91.5ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to C:\programming\python\outdoot-car-detection-satellite\runs\obb\val-11


In [ ]:
metrics_model_26_custom = model_26_custom.val(
    data=r"X:\datasets\outdoor-car-detection-satellite\data.yaml",
    split="test",
    single_cls=True,
    imgsz=1024,
    batch=16,
    device='cuda:0',
    plots=True
)

Ultralytics 8.4.154  Python-3.12.11 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
YOLO26x-obb summary (fused): 197 layers, 57,543,282 parameters, 0 gradients, 202.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2224.1755.4 MB/s, size: 1567.8 KB)
val: Scanning E:\datasets\outdoor-car-detection-satellite\test.cache... 114 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5s/it 12.2s1.8s
                   all        114       1696      0.967       0.95      0.981      0.856
Speed: 5.6ms preprocess, 91.5ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to C:\programming\python\outdoot-car-detection-satellite\runs\obb\val-13


In [63]:
print("----------- YOLO26x-OBB base -----------")
print(f"mAP50-95:  {metrics_model_26.box.map:.4f}")          # map50-95
print(f"mAP50:     {metrics_model_26.box.map50:.4f}")           # map50
print(f"mAP75:     {metrics_model_26.box.map75:.4f}")           # map75
print(f"Precision: {metrics_model_26.box.mp:.4f}")          # precision
print(f"Recall:    {metrics_model_26.box.mr:.4f}")             # recall

print("----------- YOLO26x-OBB finetuned -----------")
print(f"mAP50-95:  {metrics_model_26_custom.box.map:.4f}")
print(f"mAP50:     {metrics_model_26_custom.box.map50:.4f}")
print(f"mAP75:     {metrics_model_26_custom.box.map75:.4f}")
print(f"Precision: {metrics_model_26_custom.box.mp:.4f}")
print(f"Recall:    {metrics_model_26_custom.box.mr:.4f}")

----------- YOLO26x-OBB base -----------
mAP50-95:  0.7400
mAP50:     0.8283
mAP75:     0.8277
Precision: 0.8704
Recall:    0.9068
----------- YOLO26x-OBB finetuned -----------
mAP50-95:  0.8556
mAP50:     0.9807
mAP75:     0.9696
Precision: 0.9667
Recall:    0.9505
